# Notebook 5 — Build It, Break It, Audit It
**AI Models and Agents for Business · Week 3 · Session 5**

Zillow, the Amazon résumé screener and COMPAS all used a model that looked fine on its overall numbers. Today you see how that happens, in about fifteen minutes, on a small loan dataset.

**No API key needed.** Everything runs on Colab's own computer.

### What you will do
1. **Build** a tiny loan-approval model and note its overall accuracy.
2. **Break** it: retrain it on biased past decisions (the ✏️ YOUR TURN cell).
3. **Audit** it: compute the error rate for each group and compare.
4. (If time) Remove the column that let the bias in, and see what changes.

### Ground rules
- Run cells top to bottom. The only cells you edit are marked **✏️ YOUR TURN**.
- The data are synthetic (made up for this lab). The bank, the branches and the applicants are fictional.
- No model-evaluation jargon today: we count **how often the model is wrong, and for whom**.

In [ ]:
# Cell 1 — Setup and the applicants. Run this first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(5)
N = 4000

# Two groups of applicants. Group B is a quarter of the applicants.
group = np.where(rng.random(N) < 0.25, "B", "A")

# Which branch they applied at. Most Group B applicants apply at the Eastside branch.
eastside = np.where(group == "B", rng.random(N) < 0.85, rng.random(N) < 0.15)

income_k        = rng.normal(62, 18, N).clip(15, 200).round(0)   # income, $ thousands
debt_ratio      = rng.beta(2, 5, N).round(2)                      # monthly debt / monthly income
years_in_job    = rng.gamma(2.0, 2.5, N).clip(0, 40).round(1)
missed_payments = rng.poisson(0.6, N)                             # missed payments in the last 2 years

# What actually happened: repaid (1) or defaulted (0).
# It depends ONLY on the applicant's finances. Group and branch play no part.
score = 1.8 * (0.06 * (income_k - 60) - 5.0 * (debt_ratio - 0.3)
               + 0.15 * (years_in_job - 5) - 1.1 * missed_payments + 1.0)
repaid = (rng.random(N) < 1 / (1 + np.exp(-score))).astype(int)

loans = pd.DataFrame({
    "group": group,
    "eastside_branch": eastside.astype(int),
    "income_k": income_k,
    "debt_ratio": debt_ratio,
    "years_in_job": years_in_job,
    "missed_payments": missed_payments,
    "repaid": repaid,
})

# Hold back 30% of applicants to test the model on people it has never seen.
train, test = train_test_split(loans, test_size=0.30, random_state=5, stratify=loans["group"])

print(f"Applicants: {N:,}  (training {len(train):,}, testing {len(test):,})")
print(f"Share in Group B: {np.mean(group == 'B'):.0%}")
print()
print("Share who actually repaid, by group:")
print(loans.groupby("group")["repaid"].mean().map("{:.0%}".format).to_string())
loans.head()

Notice: **both groups repay at about the same rate.** Any difference you see later in how the model treats them did not come from the applicants.

---
## Part A — Build

The model reads five columns and predicts **approve** (expects the loan to be repaid) or **decline**. The `group` column is **not** given to the model.

We train it on what actually happened to past borrowers, then test it on applicants it has never seen.

In [ ]:
# Cell 2 — Build the model and check its overall accuracy
FEATURES = ["income_k", "debt_ratio", "years_in_job", "missed_payments", "eastside_branch"]

def build(labels, features=FEATURES):
    """Train a small model on the training applicants, using the labels you give it."""
    return LogisticRegression(max_iter=1000).fit(train[features], labels)

def accuracy(model, features=FEATURES):
    """Share of test applicants the model gets right (compared with what actually happened)."""
    return (model.predict(test[features]) == test["repaid"]).mean()

fair_model = build(train["repaid"])
print(f"Overall accuracy on new applicants: {accuracy(fair_model):.1%}")

Write that number down. It is the number a vendor would put on the first slide.

---
## Part B — Break

In real life a bank rarely knows who *would have* repaid. It knows who its loan officers **approved**. So teams often train on **past decisions** instead of outcomes, which is what Amazon did with past hiring decisions.

Suppose past officers turned down some applicants in Group B who would have repaid. `BIAS` is the share of those good Group B applicants who were wrongly turned down in the history the model learns from.

In [ ]:
# Cell 3 — ✏️ YOUR TURN: how biased were the past decisions?
# Try 0.0, 0.2, 0.4 and 0.6. Re-run this cell and the two below after each change.
BIAS = 0.4

r = np.random.default_rng(1)
past_decisions = train["repaid"].copy()
wrongly_declined = (train["group"] == "B") & (train["repaid"] == 1) & (r.random(len(train)) < BIAS)
past_decisions[wrongly_declined] = 0

biased_model = build(past_decisions)

print(f"Good Group B applicants wrongly declined in the history: {wrongly_declined.sum()} "
      f"of {((train['group'] == 'B') & (train['repaid'] == 1)).sum()}")
print()
print(f"Overall accuracy, model trained on outcomes:       {accuracy(fair_model):.1%}")
print(f"Overall accuracy, model trained on past decisions: {accuracy(biased_model):.1%}")

💬 **Discuss:** Did overall accuracy move much? If this were the only number you saw, would you approve the model for launch?

---
## Part C — Audit

Now count the mistakes **for each group**. Two kinds of mistake matter to a bank, and to its customers:

- **False rejection:** declining someone who would have repaid (a good customer turned away).
- **False approval:** approving someone who then defaulted (money lost).

In [ ]:
# Cell 4 — Error rates by group
def audit(model, features=FEATURES):
    t = test.assign(approved=model.predict(test[features]))
    rows = []
    for name, sub in [("Everyone", t), ("Group A", t[t.group == "A"]), ("Group B", t[t.group == "B"])]:
        would_repay = sub[sub.repaid == 1]
        would_default = sub[sub.repaid == 0]
        rows.append({
            "Who": name,
            "Applicants": len(sub),
            "Error rate": (sub.approved != sub.repaid).mean(),
            "False rejection rate": (would_repay.approved == 0).mean(),
            "False approval rate": (would_default.approved == 1).mean(),
        })
    table = pd.DataFrame(rows).set_index("Who")
    return table

pct = lambda df: df.style.format({c: "{:.1%}" for c in df.columns if "rate" in c})
print("Model trained on outcomes (fair labels):")
display(pct(audit(fair_model)))
print(f"Model trained on past decisions (BIAS = {BIAS}):")
display(pct(audit(biased_model)))

In [ ]:
# Cell 5 — The same audit as a picture
fair, biased = audit(fair_model), audit(biased_model)
fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(3)
ax.bar(x - 0.2, fair["False rejection rate"], width=0.4, label="Trained on outcomes")
ax.bar(x + 0.2, biased["False rejection rate"], width=0.4, label=f"Trained on past decisions (BIAS={BIAS})")
ax.set_xticks(x, fair.index)
ax.set_ylabel("Good applicants declined")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("False rejection rate, by group")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ Check before you move on
Write down two numbers for the model trained on past decisions: the **overall error rate** and **Group B's false rejection rate**. Did the overall number hide a big gap between the groups?

💬 **Discuss:**
- The model never saw the `group` column. How did it learn to treat Group B differently? (Look at the columns it *did* see. Which one stands in for group?)
- Which kind of mistake grew for Group B, and who pays for it: the bank, or the customer?
- This is the COMPAS argument in miniature: an overall number can look acceptable while one group carries most of the errors.

---
## Part D — (If time) Remove the stand-in column

The branch column was a **proxy**: it carried information about group. Amazon's engineers tried the same fix on their résumé screener by removing gendered words. Try it here.

In [ ]:
# Cell 6 — Retrain on the same biased history, without the branch column
NO_PROXY = ["income_k", "debt_ratio", "years_in_job", "missed_payments"]
no_proxy_model = build(past_decisions, features=NO_PROXY)
print(f"Without the branch column (BIAS = {BIAS}):")
display(pct(audit(no_proxy_model, features=NO_PROXY)))

💬 **Discuss:** Did the gap shrink? Real data have many columns that can stand in for a group (zip code, school, first name, the words in a résumé). Why did removing the obvious words **not** fix Amazon's screener? And why is "fix the labels" (train on outcomes, not past decisions) often the harder but better fix?

### Where should a person stay in the loop?
In one sentence: which applicants would you send to a human reviewer, and what would that cost? (Hint: the cases the model is least sure about, and the declines that would cost a customer most.)

---
## ✅ Notebook 5 complete — Reflection (submit with your notebook)

These prep you for **Quiz 2** (Session 8).

1. What was the overall accuracy before and after you broke the model? What was Group B's false rejection rate before and after?
2. In two sentences a manager could act on, report your audit finding. (A good finding names the group, the size of the gap and the kind of mistake.)
3. The model was never given the `group` column. Explain how it still learned to treat Group B differently.
4. Name one case from today (Zillow, Amazon or COMPAS) and say which part of today's notebook it resembles most, and why.

*Next session: From Prediction to Generation. Klarna's AI assistant, and Notebook 6 on the economics of escalation.*